# Tutorial 5: Parameter Calibration from Real Market Data

## From Ticker to Training in One Line

The hardest part of setting up a realistic simulation isn't the math — it's choosing the parameters. What drift should you use for AAPL? What's the right GARCH alpha for a tech stock? How many jumps per year does NVDA experience?

Spectral-Env-Core includes `estimate_params` — a calibration tool that fits all environment parameters directly from historical price data via yfinance. One ticker, one function call, ready-to-use `env_kwargs`.

---

In [ ]:
# Requires: pip install yfinance arch
from spectral_env_core import SpectralTradingEnv, estimate_params, estimate_params_multi
import numpy as np
import matplotlib.pyplot as plt

## Single-Asset Calibration

In [ ]:
# Fit parameters from NVDA's last 3 years of data
params = estimate_params("NVDA", period="3y")

print("\nFitted parameters:")
for k, v in params.items():
    print(f"  {k:>20s}: {v}")

In [ ]:
# Create environment directly from fitted params
env = SpectralTradingEnv(
    **params,
    num_steps=252,
    starting_cash=50_000,
    max_shares=200,
    max_trade_size=20,
)

# Generate and plot 20 synthetic NVDA episodes
fig, ax = plt.subplots(figsize=(14, 6))
for i in range(20):
    env.reset(seed=i)
    ax.plot(env.brownian_path[:, 0], alpha=0.5, lw=0.8)

ax.axhline(params['initial_price'], color='red', ls='--', lw=1.5, label=f"Current NVDA: ${params['initial_price']:.2f}")
ax.set_title(f'20 Synthetic NVDA Episodes (calibrated from 3y history)', fontsize=13)
ax.set_xlabel('Trading Day')
ax.set_ylabel('Price ($)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nDrift: {params['drift']:.4f} (annualised expected return)")
print(f"Volatility: {params['volatility']:.4f} (annualised)")
print(f"Jump intensity: {params['jump_intensity']:.2f} events/year")
print(f"GARCH persistence: α+β = {params['garch_alpha'] + params['garch_beta']:.4f}")

## Multi-Asset Calibration

For portfolio training, `estimate_params_multi` fits per-asset parameters AND computes the empirical correlation matrix — ready to unpack directly into the environment.

In [ ]:
# Fit a 4-asset portfolio
tickers = ["AAPL", "MSFT", "XOM", "JPM"]
multi_params = estimate_params_multi(tickers, period="3y")

print(f"\nFitted {multi_params['num_assets']}-asset parameters:")
print(f"  Prices:       {multi_params['initial_price']}")
print(f"  Drift:        {multi_params['drift']}")
print(f"  Volatility:   {multi_params['volatility']}")
print(f"  GARCH α:      {multi_params['garch_alpha']}")
print(f"  GARCH β:      {multi_params['garch_beta']}")
print(f"  Jump λ:       {multi_params['jump_intensity']}")
print(f"\n  Correlation matrix:")
corr = multi_params['correlation']
print(f"          {'  '.join(f'{t:>6s}' for t in tickers)}")
for i, t in enumerate(tickers):
    row = '  '.join(f'{corr[i,j]:+.3f}' for j in range(len(tickers)))
    print(f"  {t:>4s}  {row}")

In [ ]:
# Create multi-asset env and visualise
multi_env = SpectralTradingEnv(
    **multi_params,
    num_steps=252,
    starting_cash=200_000,
    max_shares=1000,
    max_trade_size=100,
)

multi_env.reset(seed=0)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for i, (ax, ticker) in enumerate(zip(axes.flat, tickers)):
    ax.plot(multi_env.brownian_path[:, i], lw=1.5)
    ax.set_title(f'{ticker} (σ={multi_params["volatility"][i]:.1%})')
    ax.set_ylabel('Price ($)')
    ax.grid(True, alpha=0.3)

axes[1, 0].set_xlabel('Trading Day')
axes[1, 1].set_xlabel('Trading Day')
plt.suptitle('4-Asset Portfolio: Calibrated from Real Data', fontsize=13)
plt.tight_layout()
plt.show()

## What `estimate_params` Actually Fits

Under the hood, the calibration pipeline runs:

1. **Drift & volatility** — annualised from daily log-return mean and std
2. **AR(1) phi** — OLS regression of returns on lagged returns
3. **Student-t df** — MLE fit on AR(1) residuals (controls tail fatness)
4. **GARCH(1,1) α, β** — MLE via the `arch` package (falls back to defaults if not installed)
5. **Jump parameters** — identified as returns beyond 3.5σ; intensity = count/years, mean/std from those events
6. **Correlation** — empirical correlation of aligned return series (multi-asset only)

The result is a dict that unpacks directly into `SpectralTradingEnv(**params)` — no manual tuning needed.

---

## CLI Usage

For quick terminal usage:

```bash
# Single asset
python -m spectral_env_core.est_env_params NVDA

# Multi-asset portfolio
python -m spectral_env_core.est_env_params AAPL MSFT XOM JPM --period 3y
```

Outputs a copy-pasteable `env_kwargs` block.

---

**Next tutorial:** [06 — Parallel Training](06_parallel_training.ipynb) — 5-8× speedup with SubprocVecEnv.